# Experimento adicional · 4 variantes de embeddings y docstrings

Partimos del notebook final de la práctica y cambiamos **solo dos factores**:

1. **Modelo de embeddings**
   - `BAAI/bge-small-en-v1.5` — versión final original.
   - `BAAI/bge-m3` — nuevo modelo a probar.

2. **Descripción de las tools**
   - docstrings cortos — versión final original.
   - docstrings extendidos — adaptados del nuevo fichero auxiliar.

Diseño 2×2:

| Variante | Embeddings | Docstrings |
|---|---|---|
| A | BGE-small-en-v1.5 | cortos |
| B | BGE-M3 | cortos |
| C | BGE-small-en-v1.5 | extendidos |
| D | BGE-M3 | extendidos |

Todo lo demás permanece fijo: Gemini 3.8 Flash, prompt de sistema, XBRL,
BM25/RRF, filtros de metadatos, `ToolCallLimitMiddleware`, guardrail numérico,
salida estructurada y evaluadores.

**Objetivo:** separar el efecto del embedding del efecto de las descripciones
de herramientas. No se modifica el notebook final de entrega.

## 1. Setup

In [1]:
# Preparación automática del proyecto en Google Colab.
import os
import subprocess
import sys
import shutil
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    URL_REPO = "https://github.com/rodovilllapa210/https---github.com-PRACTICA-AGENTE-RAG.git"
    CARPETA_REPO = Path("/content/MIAX_2026/Practica_Agente_RAG")

    if not (CARPETA_REPO / ".git").is_dir():
        CARPETA_REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
             URL_REPO, str(CARPETA_REPO)],
            check=True,
        )

    # Si alguno de estos ficheros se subió manualmente al panel de Colab,
    # se copia al repo. El golden oficial NO es dependencia de la entrega.
    OPCIONALES_SUBIDOS = (
        "miax_s2.py", "golden_set.jsonl",
    )
    for nombre in OPCIONALES_SUBIDOS:
        destino = CARPETA_REPO / nombre
        subido = Path("/content") / nombre
        if not destino.is_file() and subido.is_file():
            shutil.copy2(subido, destino)

    NECESARIOS = (
        "miax_s1.py", "miax_s2.py", "corpus_miax_2026.zip",
        "indice_faiss.zip", "golden_set.jsonl",
    )
    faltan = [n for n in NECESARIOS if not (CARPETA_REPO / n).is_file()]
    if faltan:
        raise FileNotFoundError(
            "Faltan archivos necesarios en el repositorio: " + ", ".join(faltan)
        )

    os.chdir(CARPETA_REPO)
    if str(CARPETA_REPO) not in sys.path:
        sys.path.insert(0, str(CARPETA_REPO))
    print("Proyecto preparado en", CARPETA_REPO)
else:
    print("Ejecución local: se usan los archivos de la carpeta actual.")


Proyecto preparado en /content/MIAX_2026/Practica_Agente_RAG


In [2]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
# Tarda alrededor de minuto y medio: mientras corre, leed la celda siguiente.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-google-genai==4.3.7 google-genai==2.10.0 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2 google-auth==2.49.0
print("Instalación terminada.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.0/148.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 179.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Clave de Gemini: secreto de Colab o variable de entorno. Nunca se imprime.
import os

HAY_CLAVE = bool(os.environ.get("GEMINI_API_KEY"))

if not HAY_CLAVE:
    try:
        from google.colab import userdata
        clave = userdata.get("GEMINI_API_KEY")
        if clave:
            os.environ["GEMINI_API_KEY"] = clave
            HAY_CLAVE = True
    except Exception:
        pass

print("GEMINI_API_KEY:", "disponible" if HAY_CLAVE else "no disponible")
if not HAY_CLAVE:
    print("Las celdas de datos y métricas funcionan; responder/evaluar requieren clave.")


GEMINI_API_KEY: disponible


In [4]:
# Preparación y verificación del corpus e índice.
import hashlib, pathlib, zipfile

PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]
URL_RESPALDO = ""          # vacio si no estan alojados
DESTINO = pathlib.Path("corpus")

CANDIDATOS = [
    pathlib.Path("."),
    pathlib.Path("/content"),
    pathlib.Path("/content/drive/MyDrive/MIAX_2026"),
    pathlib.Path("/content/drive/Shareddrives/MIAX_2026"),
]


def _sha256(ruta):
    d = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            d.update(b)
    return d.hexdigest()


def _localizar(nombre):
    for base in CANDIDATOS:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    if URL_RESPALDO:
        import urllib.request
        destino = pathlib.Path(nombre)
        urllib.request.urlretrieve(f"{URL_RESPALDO}/{nombre}", destino)
        return destino
    return None


try:
    for nombre, esperado in PAQUETES:
        origen = _localizar(nombre)
        assert origen is not None, (
            f"No encuentro {nombre}. Subelo con el panel de ficheros de "
            f"Colab (icono de carpeta a la izquierda), o monta el Drive "
            f"donde este. Buscado en: {[str(c) for c in CANDIDATOS]}"
        )
        obtenido = _sha256(origen)
        assert obtenido == esperado, (
            f"{nombre} no coincide con lo esperado: el fichero esta "
            f"corrupto o es de otra version.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}"
        )
        with zipfile.ZipFile(origen) as zf:
            zf.extractall(DESTINO)

    # Los dos manifiestos declaran el hash de chunks.jsonl. El indice se
    # construyo sobre ESE fichero: si no cuadra, el indice y sus metadatos
    # estan desalineados y el retrieval devuelve el texto equivocado sin
    # dar ningun error.
    huella = _sha256(DESTINO / "chunks.jsonl")
    for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):
        ruta = DESTINO / manifiesto
        if ruta.exists():
            assert huella in ruta.read_text(encoding="utf-8"), (
                f"chunks.jsonl no cuadra con {manifiesto}: el indice se "
                "construyo sobre otros fragmentos."
            )

    print("Corpus e indice verificados en", DESTINO.resolve())
    for p in sorted(DESTINO.rglob("*")):
        if p.is_file():
            rel = str(p.relative_to(DESTINO))
            print(f"  {rel:28s} {p.stat().st_size / 1e6:7.2f} MB")

except Exception as e:
    print("No se pudo preparar el corpus:", e)
    print("Pide los ficheros al profesor y dejalos junto al notebook.")


Corpus e indice verificados en /content/MIAX_2026/Practica_Agente_RAG/corpus
  LEEME.md                        0.00 MB
  MANIFEST.md                     0.00 MB
  chunks.jsonl                    3.80 MB
  indice/MANIFEST.md              0.00 MB
  indice/chunks_meta.parquet      1.48 MB
  indice/corpus.faiss             2.69 MB
  secciones.jsonl                 3.21 MB
  xbrl_facts.parquet              0.01 MB


In [5]:
import json
import time
import uuid
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd

secciones = pd.DataFrame([
    json.loads(x)
    for x in Path("corpus/secciones.jsonl").read_text(
        encoding="utf-8"
    ).splitlines()
    if x.strip()
])
xbrl = pd.read_parquet("corpus/xbrl_facts.parquet")
CHUNKS = [
    json.loads(x)
    for x in Path("corpus/chunks.jsonl").read_text(
        encoding="utf-8"
    ).splitlines()
    if x.strip()
]
POR_ID = {c["chunk_id"]: c for c in CHUNKS}

print(
    f"{len(secciones)} secciones · {len(CHUNKS)} chunks · "
    f"{len(xbrl)} hechos XBRL · {secciones.ticker.nunique()} compañías"
)

48 secciones · 1749 chunks · 135 hechos XBRL · 6 compañías


In [6]:
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.8-flash"

modelo = None
if HAY_CLAVE:
    try:
        # Gemini 3.8 Flash: dejamos el muestreo en la configuración
        # recomendada por el proveedor; no fijamos temperature/top_p/top_k.
        modelo = init_chat_model(MODELO)
        print("Modelo preparado:", MODELO)
        print("Sampling: model_default")
    except Exception as e:
        print(f"No se pudo crear el modelo ({type(e).__name__}: {e}).")

import pathlib
assert pathlib.Path("corpus/chunks.jsonl").is_file(), \
    "El corpus no está preparado."
assert pathlib.Path("corpus/indice/corpus.faiss").is_file(), \
    "Falta el índice FAISS."
print("§1 listo.")


Modelo preparado: google_genai:gemini-3.8-flash
Sampling: model_default
§1 listo.


## 2. Dos modelos de embeddings

In [7]:
import faiss
import torch
from sentence_transformers import SentenceTransformer

import miax_s2

MODELO_SMALL = "BAAI/bge-small-en-v1.5"
MODELO_M3 = "BAAI/bge-m3"
PREFIJO_BGE_V15 = "Represent this sentence for searching relevant passages: "

RUTA_INDICE = Path("corpus/indice")
RUTA_META = RUTA_INDICE / "chunks_meta.parquet"
RUTA_SMALL = RUTA_INDICE / "corpus.faiss"
RUTA_M3 = RUTA_INDICE / "corpus_BAAI_bge-m3.faiss"

META = pd.read_parquet(RUTA_META)

# Construimos los textos siguiendo exactamente el orden de META.
# Así el índice M3 queda alineado con los mismos metadatos que el índice original.
_por_chunk = {c["chunk_id"]: c for c in CHUNKS}
faltan = [cid for cid in META["chunk_id"] if cid not in _por_chunk]
assert not faltan, f"Hay chunk_id en META que no están en chunks.jsonl: {faltan[:5]}"
TEXTOS_META = [_por_chunk[cid]["texto"] for cid in META["chunk_id"]]

DEVICE_EMBEDDINGS = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo embeddings:", DEVICE_EMBEDDINGS)


@lru_cache(maxsize=2)
def preparar_backend(nombre_modelo: str):
    """Carga el encoder y su FAISS; crea el índice M3 la primera vez."""
    inicio = time.perf_counter()
    encoder = SentenceTransformer(nombre_modelo, device=DEVICE_EMBEDDINGS)

    if nombre_modelo == MODELO_SMALL:
        ruta = RUTA_SMALL
        if not ruta.is_file():
            raise FileNotFoundError(
                "Falta el índice original corpus/indice/corpus.faiss."
            )
        indice = faiss.read_index(str(ruta))

    elif nombre_modelo == MODELO_M3:
        ruta = RUTA_M3
        if ruta.is_file():
            indice = faiss.read_index(str(ruta))
            print("Índice M3 existente cargado:", ruta)
        else:
            print("Generando índice FAISS para BGE-M3 (solo la primera vez)...")
            vectores = encoder.encode(
                TEXTOS_META,
                batch_size=16,
                normalize_embeddings=True,
                show_progress_bar=True,
                convert_to_numpy=True,
            ).astype("float32")
            indice = faiss.IndexFlatIP(vectores.shape[1])
            indice.add(vectores)
            faiss.write_index(indice, str(ruta))
            print(
                f"Índice M3 guardado: {ruta} · "
                f"{indice.ntotal} vectores · dim={indice.d}"
            )
    else:
        raise ValueError(f"Modelo no soportado: {nombre_modelo}")

    assert indice.ntotal == len(META), (
        f"Índice desalineado: {indice.ntotal} vectores vs {len(META)} filas."
    )

    return {
        "modelo": nombre_modelo,
        "encoder": encoder,
        "indice": indice,
        "ruta": ruta,
        "dimension": indice.d,
        "carga_s": time.perf_counter() - inicio,
    }


def codificar_consulta(textos: list[str], nombre_modelo: str):
    """Replica la convención: prefijo para BGE v1.5; sin prefijo para M3."""
    backend = preparar_backend(nombre_modelo)
    if nombre_modelo == MODELO_SMALL:
        textos = [PREFIJO_BGE_V15 + t for t in textos]
    return backend["encoder"].encode(
        textos,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")


# BM25 es idéntico en las cuatro variantes.
BM25, CHUNKS_BM25 = miax_s2.montar_bm25()
KK_RRF = 60

print("Backends preparados de forma perezosa: small se carga al usar A/C; M3 al usar B/D.")

Dispositivo embeddings: cpu
Backends preparados de forma perezosa: small se carga al usar A/C; M3 al usar B/D.


## 3. Retrieval híbrido común

In [8]:
def fila_a_fragmento(fila, puntuacion: float) -> dict:
    return {
        "chunk_id": fila["chunk_id"],
        "ticker": fila["ticker"],
        "fiscal_year": int(fila["fiscal_year"]),
        "item": fila["item"],
        "texto": fila["texto"],
        "n_tokens": int(fila["n_tokens"]),
        "contiene_tabla": bool(fila["contiene_tabla"]),
        "inicio_car": int(fila["inicio_car"]),
        "fin_car": int(fila["fin_car"]),
        "puntuacion": round(float(puntuacion), 6),
    }


def buscar_hibrido(
    query: str,
    nombre_modelo: str,
    ticker: str | None = None,
    fiscal_year: int | None = None,
    item: str | None = None,
    k: int = 5,
) -> list[dict]:
    """Mismo dense + BM25/RRF de la versión final; solo cambia el embedding."""
    backend = preparar_backend(nombre_modelo)
    scores, posiciones = backend["indice"].search(
        codificar_consulta([query], nombre_modelo),
        backend["indice"].ntotal,
    )

    densos = []
    for score, idx in zip(scores[0], posiciones[0]):
        idx = int(idx)
        if idx < 0:
            continue
        fila = META.iloc[idx]
        if ticker is not None and fila["ticker"] != ticker:
            continue
        if fiscal_year is not None and int(fila["fiscal_year"]) != int(fiscal_year):
            continue
        if item is not None and fila["item"] != item:
            continue
        densos.append(fila_a_fragmento(fila, score))

    if not densos:
        return []

    rank_dense = {f["chunk_id"]: i for i, f in enumerate(densos, 1)}
    permitidos = set(rank_dense)

    bm_scores = BM25.get_scores(miax_s2.tokenizar(query))
    rank_bm25, puesto = {}, 0
    for idx in np.argsort(-bm_scores):
        cid = CHUNKS_BM25[int(idx)]["chunk_id"]
        if cid in permitidos:
            puesto += 1
            rank_bm25[cid] = puesto

    rrf = {
        cid: (
            1 / (KK_RRF + rd)
            + 1 / (KK_RRF + rank_bm25.get(cid, 10**6))
        )
        for cid, rd in rank_dense.items()
    }
    por_id = {f["chunk_id"]: f for f in densos}

    salida = []
    for cid in sorted(rrf, key=rrf.get, reverse=True)[:int(k)]:
        f = dict(por_id[cid])
        f["puntuacion"] = round(float(rrf[cid]), 6)
        salida.append(f)
    return salida


def formatear_fragmentos(fragmentos: list[dict]) -> str:
    if not fragmentos:
        return "Sin resultados para esa consulta con esos filtros."
    return "\n\n---\n\n".join(
        f"[{f['chunk_id']}] {f['ticker']} FY{f['fiscal_year']} "
        f"Item {f['item']} (RRF {f['puntuacion']:.6f})\n{f['texto']}"
        for f in fragmentos
    )

## 4. Tools: docstrings cortos vs extendidos

In [9]:
from langchain.tools import tool

CONCEPTOS_XBRL = (
    "Assets, CashAndCashEquivalentsAtCarryingValue, EarningsPerShareBasic, "
    "EarningsPerShareDiluted, GrossProfit, Liabilities, "
    "NetCashProvidedByUsedInOperatingActivities, NetIncomeLoss, "
    "OperatingIncomeLoss, ResearchAndDevelopmentExpense, "
    "RevenueFromContractWithCustomerExcludingAssessedTax, "
    "StockholdersEquity, Revenues"
)

DOCS_CORTOS = {
    "list_available":
        "Lista compañías, ejercicios y secciones disponibles en el corpus.",

    "get_xbrl_fact":
        """Devuelve el valor exacto de una magnitud financiera reportada en XBRL.

Args:
    ticker: Símbolo bursátil.
    fiscal_year: Ejercicio fiscal.
    concept: Concepto US-GAAP.""",

    "search_filings":
        """Busca texto relevante en los 10-K mediante dense + BM25/RRF.

Args:
    query: Consulta, preferiblemente en inglés.
    ticker: Filtro opcional por compañía.
    fiscal_year: Filtro opcional por ejercicio.
    item: Filtro opcional: '1A', '7', '7A' u '8'.
    k: Número de fragmentos.""",

    "read_section":
        "Devuelve el texto completo de una sección; úsala solo como fallback.",
}

DOCS_LARGOS = {
    "list_available":
        """Devuelve qué compañías, ejercicios y secciones existen en el corpus.

Úsala cuando no estés seguro de que la empresa, el ejercicio o la sección
mencionados por el usuario estén disponibles. El corpus es limitado y esta
herramienta permite comprobar su cobertura antes de buscar información.""",

    "get_xbrl_fact":
        f"""Devuelve el valor EXACTO de una magnitud financiera estandarizada
reportada por una compañía en XBRL.

Es la fuente autorizada para magnitudes contables estructuradas como ingresos,
activos, beneficio neto, beneficio bruto, I+D, caja o patrimonio. Úsala para
esas cifras en lugar de leerlas de la prosa del informe.

NO la uses para guidance, porcentajes narrativos, mix de negocio, explicaciones
de la dirección u otras cifras que solo aparezcan en el texto: en esos casos
usa search_filings.

Args:
    ticker: Símbolo bursátil. Corpus: AAPL, AMZN, GOOGL, META, MSFT y NVDA.
    fiscal_year: Ejercicio fiscal. Corpus: 2024 y 2025.
    concept: Concepto US-GAAP. Conceptos disponibles: {CONCEPTOS_XBRL}.

Devuelve el valor con su unidad y fecha de cierre, o un aviso explícito si ese
concepto no está disponible para la compañía y ejercicio solicitados.""",

    "search_filings":
        """Busca fragmentos relevantes dentro de los informes 10-K mediante
retrieval híbrido: búsqueda semántica densa + BM25, fusionadas con RRF.

Úsala para información cualitativa o narrativa: riesgos, estrategia, litigios,
explicaciones de la dirección, guidance, porcentajes narrativos, mix de negocio
y causas de variaciones entre ejercicios.

Para magnitudes contables estandarizadas disponibles en XBRL, usa
get_xbrl_fact. Las consultas funcionan mejor en inglés porque los 10-K están
redactados en inglés.

Args:
    query: Qué buscar, en lenguaje natural y preferiblemente en inglés.
    ticker: Filtro opcional por compañía: AAPL, AMZN, GOOGL, META, MSFT o NVDA.
    fiscal_year: Filtro opcional por ejercicio: 2024 o 2025.
    item: Filtro opcional por sección: '1A' riesgos, '7' MD&A,
          '7A' riesgo de mercado, '8' estados financieros.
    k: Número de fragmentos a devolver; 5 es el valor recomendado.

Devuelve fragmentos con chunk_id para que la respuesta pueda citarse y
verificarse.""",

    "read_section":
        """Devuelve el TEXTO COMPLETO de una sección concreta de un 10-K.

Es una herramienta cara porque una sección puede contener decenas de miles de
tokens. Úsala solo como último recurso, cuando search_filings no haya devuelto
contexto suficiente y necesites leer una sección completa.

Args:
    ticker: Símbolo bursátil.
    fiscal_year: Ejercicio fiscal.
    item: Sección del 10-K: '1A', '7', '7A' u '8'.""",
}


def crear_herramientas(nombre_modelo: str, docs_largos: bool):
    """Crea las cuatro tools sin cambiar su comportamiento ni sus firmas."""
    docs = DOCS_LARGOS if docs_largos else DOCS_CORTOS

    def list_available() -> str:
        lineas = []
        for ticker, filas in secciones.groupby("ticker", sort=True):
            years = sorted(filas.fiscal_year.astype(int).unique())
            items = sorted(filas.item.astype(str).unique())
            lineas.append(f"{ticker}: FY{years} · Items {items}")
        return "\n".join(lineas)

    def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
        filas = xbrl[
            (xbrl.ticker == ticker)
            & (xbrl.fiscal_year == int(fiscal_year))
            & (xbrl.concept == concept)
        ]
        if filas.empty:
            disponibles = sorted(
                xbrl[
                    (xbrl.ticker == ticker)
                    & (xbrl.fiscal_year == int(fiscal_year))
                ].concept.unique()
            )
            return (
                f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
                f"Conceptos disponibles: {', '.join(disponibles) or 'ninguno'}."
            )
        f = filas.iloc[0]
        return (
            f"{ticker} FY{fiscal_year} {concept} = {f.value:,.0f} {f.unit} "
            f"(cierre {f.period_end}, {f.form})"
        )

    def search_filings(
        query: str,
        ticker: str | None = None,
        fiscal_year: int | None = None,
        item: str | None = None,
        k: int = 5,
    ) -> str:
        return formatear_fragmentos(
            buscar_hibrido(
                query,
                nombre_modelo=nombre_modelo,
                ticker=ticker,
                fiscal_year=fiscal_year,
                item=item,
                k=k,
            )
        )

    def read_section(ticker: str, fiscal_year: int, item: str) -> str:
        filas = secciones[
            (secciones.ticker == ticker)
            & (secciones.fiscal_year == int(fiscal_year))
            & (secciones.item == item)
        ]
        if filas.empty:
            return f"No hay Item {item} de {ticker} FY{fiscal_year}."
        return filas.iloc[0].texto

    funciones = {
        "list_available": list_available,
        "get_xbrl_fact": get_xbrl_fact,
        "search_filings": search_filings,
        "read_section": read_section,
    }

    herramientas = []
    for nombre, funcion in funciones.items():
        funcion.__name__ = nombre
        funcion.__doc__ = docs[nombre]
        herramientas.append(tool(nombre)(funcion))

    return herramientas

## 5. Agente común a las cuatro variantes

In [10]:
from typing import Literal

from pydantic import BaseModel, model_validator
from langchain.agents import create_agent
from langchain.agents.middleware import AgentState, ToolCallLimitMiddleware, after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

SYSTEM = """Eres un analista financiero que responde preguntas sobre informes
10-K usando ÚNICAMENTE las herramientas disponibles.

- Magnitudes contables estandarizadas disponibles en XBRL:
  usa get_xbrl_fact.
- Riesgos, estrategia, guidance, porcentajes narrativos y explicaciones:
  usa search_filings.
- Una pregunta puede combinar XBRL y texto.
- El corpus está en inglés: formula las búsquedas en inglés.
- Si el dato no está en el corpus, dilo; no lo estimes.
- Si la pregunta es solo XBRL, no confirmes la cifra con texto.
- Usa search_filings con k=5 por defecto y deja de buscar cuando haya
  evidencia suficiente; read_section es último recurso.
- En comparativas, `cifra` es siempre el valor de la magnitud principal en
  el ejercicio final; diferencias y porcentajes van en `respuesta`.
- Si `fuente` es "texto" o "ambas", `cita` y `chunk_id` son obligatorios y
  deben corresponder al mismo fragmento recuperado.
"""


class RespuestaFinanciera(BaseModel):
    respuesta: str
    cifra: float | None = None
    unidad: str | None = None
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"]
    cita: str | None = None
    chunk_id: str | None = None

    @model_validator(mode="after")
    def trazabilidad(self):
        if self.fuente in {"texto", "ambas"}:
            if not self.cita or not self.chunk_id:
                raise ValueError(
                    "fuente texto/ambas exige cita y chunk_id."
                )
        return self


MARCA_GUARDRAIL = "VERIFICACIÓN AUTOMÁTICA"
TOLERANCIA_GUARDRAIL = 0.01


def _contenido(m) -> str:
    return (
        str(m.get("content", ""))
        if isinstance(m, dict)
        else str(getattr(m, "content", ""))
    )


@after_model(can_jump_to=["model"])
def verificar_cifras_contra_xbrl(
    state: AgentState, runtime: Runtime
) -> dict | None:
    """Mismo guardrail de la versión final."""
    r = state.get("structured_response")
    if (
        r is None
        or getattr(r, "cifra", None) is None
        or getattr(r, "fuente", None) not in {"xbrl", "ambas"}
    ):
        return None

    ticker = getattr(r, "ticker", None)
    ejercicio = getattr(r, "ejercicio", None)
    if ticker is None or ejercicio is None:
        return None

    if any(
        MARCA_GUARDRAIL in _contenido(m)
        for m in state.get("messages", [])
    ):
        return None

    hechos = xbrl[
        (xbrl.ticker == ticker)
        & (xbrl.fiscal_year == int(ejercicio))
    ]
    if hechos.empty:
        return None

    afirmada = float(r.cifra)
    if any(
        miax_s2.cuadra(
            afirmada, float(v), TOLERANCIA_GUARDRAIL
        )
        for v in hechos.value
    ):
        return None

    disponibles = "; ".join(
        f"{f.concept}={float(f.value):,.0f} {f.unit}"
        for f in hechos.itertuples()
    )
    return {
        "messages": [{
            "role": "user",
            "content": (
                f"{MARCA_GUARDRAIL}: {afirmada:,.6g} no coincide con XBRL "
                f"de {ticker} FY{ejercicio} (tolerancia 1%). "
                f"Valores disponibles: {disponibles}. Corrige la respuesta."
            ),
        }],
        "jump_to": "model",
    }


VARIANTES = {
    "A": {
        "nombre": "A · Small + docs cortos",
        "embedding": MODELO_SMALL,
        "docs_largos": False,
    },
    "B": {
        "nombre": "B · M3 + docs cortos",
        "embedding": MODELO_M3,
        "docs_largos": False,
    },
    "C": {
        "nombre": "C · Small + docs largos",
        "embedding": MODELO_SMALL,
        "docs_largos": True,
    },
    "D": {
        "nombre": "D · M3 + docs largos",
        "embedding": MODELO_M3,
        "docs_largos": True,
    },
}

AGENTES = {}
TOOLS_VARIANTE = {}


def crear_agente_variante(codigo: str):
    cfg = VARIANTES[codigo]
    herramientas = crear_herramientas(
        cfg["embedding"],
        cfg["docs_largos"],
    )
    TOOLS_VARIANTE[codigo] = herramientas

    return create_agent(
        model=modelo,
        tools=herramientas,
        system_prompt=SYSTEM,
        response_format=RespuestaFinanciera,
        checkpointer=InMemorySaver(),
        middleware=[
            ToolCallLimitMiddleware(
                run_limit=11,
                exit_behavior="end",
            ),
            verificar_cifras_contra_xbrl,
        ],
    )


if HAY_CLAVE:
    for codigo in VARIANTES:
        AGENTES[codigo] = crear_agente_variante(codigo)
    print("Agentes A/B/C/D preparados.")
else:
    print("Sin clave: los agentes no se han creado.")

Agentes A/B/C/D preparados.


## 6. Golden set y evaluadores

In [11]:
RUTA_GOLDEN = Path("golden_set.jsonl")
if not RUTA_GOLDEN.is_file():
    raise FileNotFoundError("Falta golden_set.jsonl.")


def leer_jsonl(ruta):
    return [
        json.loads(x)
        for x in Path(ruta).read_text(encoding="utf-8").splitlines()
        if x.strip()
    ]


golden = leer_jsonl(RUTA_GOLDEN)

CAMPOS = {
    "id", "pregunta", "familia", "ticker", "fiscal_year",
    "respuesta_esperada", "cifra_esperada", "unidad", "concept_xbrl",
    "item_esperado", "ancla_texto", "ancla_inicio", "ancla_fin",
    "chunk_id_esperado", "herramienta_esperada", "autor",
}


def validar(preguntas, exigir_20=True):
    problemas, vistos = [], set()
    tickers = set(secciones.ticker)

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        faltan = CAMPOS - set(p)
        if faltan:
            problemas.append(f"{pid}: faltan {sorted(faltan)}")
            continue
        if pid in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(pid)
        if p["familia"] not in {"numerica", "extractiva", "comparativa"}:
            problemas.append(f"{pid}: familia inválida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: ticker fuera del corpus")
        if p["familia"] in {"numerica", "comparativa"}:
            if p["cifra_esperada"] is None:
                problemas.append(f"{pid}: falta cifra_esperada")
        if p["familia"] in {"extractiva", "comparativa"}:
            if not p["ancla_texto"]:
                problemas.append(f"{pid}: falta ancla_texto")
        if not p["herramienta_esperada"]:
            problemas.append(f"{pid}: falta herramienta_esperada")

    if exigir_20:
        if len(preguntas) != 20:
            problemas.append(
                f"se exigen 20 preguntas; hay {len(preguntas)}"
            )
        if sum(
            p["familia"] == "comparativa"
            for p in preguntas
        ) < 6:
            problemas.append("se exigen al menos 6 comparativas")

    return problemas


problemas = validar(golden)
assert not problemas, "\n".join(problemas)

print(
    f"Golden OK: {len(golden)} preguntas · "
    f"{sum(g['familia']=='comparativa' for g in golden)} comparativas"
)

Golden OK: 20 preguntas · 6 comparativas


In [12]:
PRECIO_INPUT_USD_M = 0.75
PRECIO_OUTPUT_USD_M = 3.75
NOMBRES_HERRAMIENTAS = {
    "list_available",
    "get_xbrl_fact",
    "search_filings",
    "read_section",
}


def responder_variante(codigo: str, pregunta: str):
    if codigo not in AGENTES:
        raise RuntimeError(
            f"Agente {codigo} no disponible. Configura GEMINI_API_KEY."
        )
    return AGENTES[codigo].invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={
            "configurable": {
                "thread_id": f"{codigo}-{uuid.uuid4()}"
            }
        },
    )


def cita_correcta(item, resultado):
    r = resultado.get("structured_response")
    if r is None:
        return None if item["familia"] == "numerica" else False
    if not r.chunk_id:
        return None if item["familia"] == "numerica" else False
    if r.chunk_id not in POR_ID or not r.cita:
        return False

    return (
        miax_s2.normalizar(str(r.cita))[:120]
        in miax_s2.normalizar(POR_ID[r.chunk_id]["texto"])
    )


def cifra_coincide_xbrl(item, resultado):
    esperada = item.get("cifra_esperada")
    if esperada is None:
        return None
    r = resultado.get("structured_response")
    return bool(
        r is not None
        and r.cifra is not None
        and miax_s2.cuadra(
            float(r.cifra), float(esperada), 0.01
        )
    )


def uso_la_tool_correcta(item, resultado):
    usadas = set(miax_s2.herramientas_usadas(resultado))
    return set(item["herramienta_esperada"]).issubset(usadas)


def recall_variante(item, codigo: str):
    if not item.get("ancla_texto"):
        return None
    cfg = VARIANTES[codigo]
    recuperados = buscar_hibrido(
        item["pregunta"],
        nombre_modelo=cfg["embedding"],
        ticker=item["ticker"],
        fiscal_year=item["fiscal_year"],
        item=item["item_esperado"],
        k=5,
    )
    return miax_s2.acierta(item, recuperados)


def coste(resultado):
    entrada, salida = miax_s2.tokens_de(resultado)
    return (
        entrada * PRECIO_INPUT_USD_M
        + salida * PRECIO_OUTPUT_USD_M
    ) / 1e6


def respuesta_dict(resultado):
    r = resultado.get("structured_response")
    return (
        r.model_dump()
        if hasattr(r, "model_dump")
        else r
    )


def trayectoria(resultado):
    salidas = {
        getattr(m, "tool_call_id", None):
            str(getattr(m, "content", ""))
        for m in resultado.get("messages", [])
        if getattr(m, "tool_call_id", None)
    }

    out = []
    for m in resultado.get("messages", []):
        for c in getattr(m, "tool_calls", []) or []:
            if c.get("name") in NOMBRES_HERRAMIENTAS:
                out.append({
                    "name": c["name"],
                    "args": c.get("args", {}),
                    "output": salidas.get(c.get("id")),
                })
    return out


def evaluar_una(caso, codigo: str):
    t0 = time.perf_counter()
    error = None

    try:
        resultado = responder_variante(
            codigo, caso["pregunta"]
        )
    except Exception as exc:
        resultado = {
            "messages": [],
            "structured_response": None,
        }
        error = f"{type(exc).__name__}: {exc}"

    latencia = time.perf_counter() - t0

    ev = {
        "cita": cita_correcta(caso, resultado),
        "cifra": cifra_coincide_xbrl(caso, resultado),
        "trayectoria": uso_la_tool_correcta(caso, resultado),
        "recall@5": recall_variante(caso, codigo),
    }

    aplicables = [
        v
        for k, v in ev.items()
        if k != "recall@5" and v is not None
    ]

    return {
        "variante": codigo,
        "id": caso["id"],
        "familia": caso["familia"],
        "pregunta": caso["pregunta"],
        "respuesta": respuesta_dict(resultado),
        "trayectoria": trayectoria(resultado),
        "evaluacion": ev,
        "acierto": (
            bool(aplicables)
            and all(aplicables)
            and error is None
        ),
        "coste_usd": coste(resultado),
        "latencia_s": latencia,
        "n_llamadas": len(
            miax_s2.herramientas_usadas(resultado)
        ),
        "error": error,
    }


def evaluar_variante(codigo: str):
    registros = [
        evaluar_una(caso, codigo)
        for caso in golden
    ]

    tabla = pd.DataFrame([{
        "variante": codigo,
        "id": r["id"],
        "familia": r["familia"],
        "acierto": r["acierto"],
        "cita": r["evaluacion"]["cita"],
        "cifra": r["evaluacion"]["cifra"],
        "trayectoria": r["evaluacion"]["trayectoria"],
        "recall": r["evaluacion"]["recall@5"],
        "coste_usd": r["coste_usd"],
        "latencia_s": r["latencia_s"],
        "llamadas": r["n_llamadas"],
        "error": r["error"],
    } for r in registros])

    return registros, tabla

## 7. Retrieval aislado: Small vs M3

In [13]:
def comparar_retrieval():
    casos = [g for g in golden if g.get("ancla_texto")]
    filas = []

    for nombre_modelo in [MODELO_SMALL, MODELO_M3]:
        hits = []
        tiempos = []

        for g in casos:
            t0 = time.perf_counter()
            recuperados = buscar_hibrido(
                g["pregunta"],
                nombre_modelo=nombre_modelo,
                ticker=g["ticker"],
                fiscal_year=g["fiscal_year"],
                item=g["item_esperado"],
                k=5,
            )
            tiempos.append(time.perf_counter() - t0)
            hits.append(miax_s2.acierta(g, recuperados))

        backend = preparar_backend(nombre_modelo)
        filas.append({
            "embedding": nombre_modelo,
            "recall@5": sum(hits) / len(hits),
            "hits": f"{sum(hits)}/{len(hits)}",
            "latencia_query_media_s": np.mean(tiempos),
            "dimension": backend["dimension"],
            "indice": str(backend["ruta"]),
        })

    return pd.DataFrame(filas)


comparacion_retrieval = comparar_retrieval()
display(
    comparacion_retrieval.style.format({
        "recall@5": "{:.1%}",
        "latencia_query_media_s": "{:.3f}",
    })
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Generando índice FAISS para BGE-M3 (solo la primera vez)...


Batches:   0%|          | 0/110 [00:00<?, ?it/s]

Índice M3 guardado: corpus/indice/corpus_BAAI_bge-m3.faiss · 1749 vectores · dim=1024


,embedding,recall@5,hits,latencia_query_media_s,dimension,indice
0,BAAI/bge-small-en-v1.5,61.5%,8/13,0.618,384,corpus/indice/corpus.faiss
1,BAAI/bge-m3,61.5%,8/13,36.871,1024,corpus/indice/corpus_BAAI_bge-m3.faiss


## 8. Ejecutar las cuatro variantes

In [14]:
CARPETA_RESULTADOS = Path("resultados_experimento_m3")
CARPETA_RESULTADOS.mkdir(exist_ok=True)

# Si un resultado ya existe, se reutiliza. Cambia a True solo si quieres
# repetir las 80 preguntas de las cuatro variantes.
FORZAR_REEJECUCION = False
VARIANTES_A_EJECUTAR = ["A", "B", "C", "D"]


def ejecutar_y_guardar_variante(codigo: str):
    p_json = CARPETA_RESULTADOS / f"{codigo}_registros.jsonl"
    p_csv = CARPETA_RESULTADOS / f"{codigo}_tabla.csv"

    if (
        p_json.is_file()
        and p_csv.is_file()
        and not FORZAR_REEJECUCION
    ):
        registros = leer_jsonl(p_json)
        tabla = pd.read_csv(p_csv)
        print(
            f"{codigo}: resultados existentes cargados; "
            "no se llamó a Gemini."
        )
        return registros, tabla

    print(
        f"\n{codigo} · {VARIANTES[codigo]['nombre']} "
        "→ ejecutando 20 preguntas..."
    )
    registros, tabla = evaluar_variante(codigo)

    with p_json.open("w", encoding="utf-8") as f:
        for r in registros:
            f.write(
                json.dumps(
                    r,
                    ensure_ascii=False,
                    allow_nan=False,
                )
                + "\n"
            )

    tabla.to_csv(p_csv, index=False)
    print(
        f"{codigo}: {int(tabla.acierto.sum())}/{len(tabla)} "
        "aciertos guardados."
    )
    return registros, tabla


RESULTADOS = {}
for codigo in VARIANTES_A_EJECUTAR:
    RESULTADOS[codigo] = ejecutar_y_guardar_variante(codigo)


A · A · Small + docs cortos → ejecutando 20 preguntas...
A: 20/20 aciertos guardados.

B · B · M3 + docs cortos → ejecutando 20 preguntas...
B: 19/20 aciertos guardados.

C · C · Small + docs largos → ejecutando 20 preguntas...
C: 20/20 aciertos guardados.

D · D · M3 + docs largos → ejecutando 20 preguntas...
D: 20/20 aciertos guardados.


## 9. Comparación A/B/C/D

In [15]:
def resumen_variante(codigo: str, tabla: pd.DataFrame):
    cfg = VARIANTES[codigo]
    return {
        "variante": codigo,
        "configuración": cfg["nombre"],
        "embedding": cfg["embedding"],
        "docstrings": (
            "largos" if cfg["docs_largos"] else "cortos"
        ),
        "accuracy": tabla.acierto.mean(),
        "recall@5": tabla.recall.dropna().mean(),
        "cita": tabla.cita.dropna().mean(),
        "cifra": tabla.cifra.dropna().mean(),
        "trayectoria": tabla.trayectoria.dropna().mean(),
        "coste_medio_usd": tabla.coste_usd.mean(),
        "latencia_media_s": tabla.latencia_s.mean(),
        "llamadas_por_pregunta": tabla.llamadas.mean(),
    }


comparacion = pd.DataFrame([
    resumen_variante(codigo, RESULTADOS[codigo][1])
    for codigo in VARIANTES_A_EJECUTAR
])

display(
    comparacion.style
    .format({
        "accuracy": "{:.1%}",
        "recall@5": "{:.1%}",
        "cita": "{:.1%}",
        "cifra": "{:.1%}",
        "trayectoria": "{:.1%}",
        "coste_medio_usd": "${:.4f}",
        "latencia_media_s": "{:.2f}",
        "llamadas_por_pregunta": "{:.2f}",
    })
    .highlight_max(
        subset=[
            "accuracy", "recall@5",
            "cita", "cifra", "trayectoria",
        ],
        axis=0,
    )
    .highlight_min(
        subset=[
            "coste_medio_usd",
            "latencia_media_s",
            "llamadas_por_pregunta",
        ],
        axis=0,
    )
)

# Accuracy por familia.
familias = []
for codigo in VARIANTES_A_EJECUTAR:
    tabla = RESULTADOS[codigo][1]
    for familia, grupo in tabla.groupby("familia"):
        familias.append({
            "variante": codigo,
            "familia": familia,
            "aciertos": int(grupo.acierto.sum()),
            "n": len(grupo),
            "accuracy": grupo.acierto.mean(),
        })

por_familia = pd.DataFrame(familias)
display(
    por_familia.pivot(
        index="variante",
        columns="familia",
        values="accuracy",
    ).style.format("{:.1%}")
)

# Matriz por pregunta: permite ver qué casos cambian entre variantes.
por_pregunta = None
for codigo in VARIANTES_A_EJECUTAR:
    t = RESULTADOS[codigo][1][["id", "acierto"]].copy()
    t = t.rename(columns={"acierto": codigo})
    por_pregunta = (
        t if por_pregunta is None
        else por_pregunta.merge(t, on="id", how="outer")
    )

display(por_pregunta)

comparacion.to_csv(
    CARPETA_RESULTADOS / "comparacion_4_variantes.csv",
    index=False,
)
por_familia.to_csv(
    CARPETA_RESULTADOS / "comparacion_por_familia.csv",
    index=False,
)
por_pregunta.to_csv(
    CARPETA_RESULTADOS / "comparacion_por_pregunta.csv",
    index=False,
)
comparacion_retrieval.to_csv(
    CARPETA_RESULTADOS / "comparacion_retrieval_small_vs_m3.csv",
    index=False,
)

,variante,configuración,embedding,docstrings,accuracy,recall@5,cita,cifra,trayectoria,coste_medio_usd,latencia_media_s,llamadas_por_pregunta
0,A,A · Small + docs cortos,BAAI/bge-small-en-v1.5,cortos,100.0%,61.5%,100.0%,100.0%,100.0%,$0.0116,7.93,3.90
1,B,B · M3 + docs cortos,BAAI/bge-m3,cortos,95.0%,61.5%,92.3%,92.3%,100.0%,$0.0144,8.66,4.25
2,C,C · Small + docs largos,BAAI/bge-small-en-v1.5,largos,100.0%,61.5%,100.0%,100.0%,100.0%,$0.0132,7.68,3.80
3,D,D · M3 + docs largos,BAAI/bge-m3,largos,100.0%,61.5%,100.0%,100.0%,100.0%,$0.0143,8.05,3.90


familia,comparativa,extractiva,numerica
variante,,,
A,100.0%,100.0%,100.0%
B,83.3%,100.0%,100.0%
C,100.0%,100.0%,100.0%
D,100.0%,100.0%,100.0%


,id,A,B,C,D
0,C1,True,True,True,True
1,C2,True,True,True,True
2,C3,True,False,True,True
3,C4,True,True,True,True
4,C5,True,True,True,True
5,C6,True,True,True,True
6,E1,True,True,True,True
7,E2,True,True,True,True
8,E3,True,True,True,True
9,E4,True,True,True,True


## 10. Cómo interpretar el experimento

Las comparaciones causales relevantes son:

- **B − A:** efecto de cambiar Small → M3 con docstrings cortos.
- **C − A:** efecto de ampliar docstrings manteniendo Small.
- **D − B:** efecto de ampliar docstrings manteniendo M3.
- **D − C:** efecto de cambiar Small → M3 con docstrings largos.
- **D − A:** efecto combinado de ambos cambios.

No conviene concluir que una variante es mejor solo por `accuracy`.
Hay que mirar también `recall@5`, routing de tools, coste, latencia y estabilidad
por familias.

La variante A es el control equivalente a nuestra arquitectura final. Debido al
muestreo del LLM, una nueva ejecución puede no reproducir exactamente el mismo
20/20 histórico; por eso la comparación principal es entre A/B/C/D ejecutadas
en la misma sesión.

In [16]:
# Opcional: comprimir todos los resultados al terminar.
import shutil

zip_creado = shutil.make_archive(
    "resultados_experimento_m3",
    "zip",
    root_dir=CARPETA_RESULTADOS,
)
print("ZIP creado:", zip_creado)

try:
    from google.colab import files
    print(
        "En Colab puedes descargarlo con:\n"
        "files.download('resultados_experimento_m3.zip')"
    )
except ImportError:
    pass

ZIP creado: /content/MIAX_2026/Practica_Agente_RAG/resultados_experimento_m3.zip
En Colab puedes descargarlo con:
files.download('resultados_experimento_m3.zip')


In [17]:
files.download('resultados_experimento_m3.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>